**Differential TE Expression in MPNST**

This notebook analyses the final MPNST TE count matrix.

Primary comparison: PRC2_loss MPNST vs PRC2_intact MPNST

Preferred input is a raw TE counts matrix, because differential count methods such as DESeq2/edgeR model count data directly

In [ ]:
# Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(stringr)
library(purrr)

project_root <- "/home/kennes38/ResearchProject"
mpnst_root <- file.path(project_root, "25_MPNST_TE_analysis")
dir.create(mpnst_root, showWarnings = FALSE, recursive = TRUE)
setwd(mpnst_root)
manifest_dir <- file.path(mpnst_root, "manifests")
counts_dir <- file.path(mpnst_root, "te_counts")
results_dir <- file.path(mpnst_root, "differential_expression")
dir.create(results_dir, showWarnings = FALSE, recursive = TRUE)

sample_manifest_file <- file.path(manifest_dir, "MPNST_RNAseq_sample_manifest.csv")
te_counts_file <- file.path(counts_dir, "MPNST_TE_counts_matrix.csv")

file.exists(sample_manifest_file)
file.exists(te_counts_file)

In [ ]:
# Load metadata and TE counts

if (!file.exists(sample_manifest_file)) stop("Missing sample manifest: ", sample_manifest_file)
if (!file.exists(te_counts_file)) stop("Missing TE count matrix: ", te_counts_file)

metadata <- read_csv(sample_manifest_file, show_col_types = FALSE) %>%
  mutate(
    sample_id = as.character(sample_id),
    PRC2_status = factor(PRC2_status, levels = c("PRC2_intact", "PRC2_loss")),
    tumour_type = as.factor(tumour_type)
  )

# Expected format: first column - feature_id, remaining columns - sample IDs.

te_counts <- read_csv(te_counts_file, show_col_types = FALSE)

cat("Metadata rows:", nrow(metadata), "\n")
cat("TE count rows:", nrow(te_counts), "\n")

head(metadata)
head(te_counts[, 1:min(6, ncol(te_counts))])

In [ ]:
# Align count matrix to metadata

feature_col <- colnames(te_counts)[1]

count_mat <- te_counts %>%
  column_to_rownames(feature_col) %>%
  as.matrix()

# Keep only samples present in both metadata and count matrix.
common_samples <- intersect(metadata$sample_id, colnames(count_mat))

analysis_meta <- metadata %>%
  filter(sample_id %in% common_samples) %>%
  filter(PRC2_status %in% c("PRC2_intact", "PRC2_loss")) %>%
  arrange(PRC2_status, sample_id)

analysis_counts <- count_mat[, analysis_meta$sample_id, drop = FALSE]

stopifnot(identical(colnames(analysis_counts), analysis_meta$sample_id))

cat("Samples included:", ncol(analysis_counts), "\n")
analysis_meta %>% count(PRC2_status)

In [ ]:
# Basic count filtering and quality control

# Remove TE features with almost no counts across all samples -> reduces noise and multiple testing burden
keep <- rowSums(analysis_counts >= 10) >= max(2, floor(0.1 * ncol(analysis_counts)))
filtered_counts <- analysis_counts[keep, , drop = FALSE]

cat("TE features before filtering:", nrow(analysis_counts), "\n")
cat("TE features after filtering:", nrow(filtered_counts), "\n")

library_size <- colSums(filtered_counts)
qc_tbl <- analysis_meta %>%
  mutate(library_size = library_size[sample_id])

write_csv(qc_tbl, file.path(results_dir, "MPNST_TE_count_QC_sample_summary.csv"))

qc_tbl %>% arrange(library_size)

In [ ]:
# DESeq2 Differential Expression

if (!requireNamespace("DESeq2", quietly = TRUE)) {
  stop("DESeq2 is not installed in this R environment. Ask about installing/loading Bioconductor DESeq2, or use edgeR/limma-voom.")
}

library(DESeq2)

# DESeq2 requires integer counts
filtered_counts_integer <- round(filtered_counts)
storage.mode(filtered_counts_integer) <- "integer"

# Simple first-pass model
design_formula <- ~ PRC2_status

dds <- DESeqDataSetFromMatrix(
  countData = filtered_counts_integer,
  colData = as.data.frame(analysis_meta %>% column_to_rownames("sample_id")),
  design = design_formula
)

dds <- DESeq(dds)

res <- results(dds, contrast = c("PRC2_status", "PRC2_loss", "PRC2_intact"))

res_tbl <- as.data.frame(res) %>%
  rownames_to_column("feature_id") %>%
  as_tibble() %>%
  arrange(padj)

write_csv(res_tbl, file.path(results_dir, "MPNST_PRC2_loss_vs_intact_TE_DESeq2_results.csv"))

res_tbl %>% slice_head(n = 30)

In [ ]:
# Significance Summary and Candidate ERV/LTR Hits

# Summary of all TE results
sig_summary <- res_tbl %>%
  summarise(
    n_tested = sum(!is.na(pvalue)),
    n_fdr_0_05 = sum(padj < 0.05, na.rm = TRUE),
    n_fdr_0_10 = sum(padj < 0.10, na.rm = TRUE),
    n_nominal_0_05 = sum(pvalue < 0.05, na.rm = TRUE),
    n_up_fdr_0_05 = sum(padj < 0.05 & log2FoldChange > 0, na.rm = TRUE),
    n_down_fdr_0_05 = sum(padj < 0.05 & log2FoldChange < 0, na.rm = TRUE)
  )

write_csv(sig_summary, file.path(results_dir, "MPNST_TE_DESeq2_significance_summary.csv"))
sig_summary

# Candidate ERV/LTR families from your TCGA analyses
tcga_candidate_hits <- c("LTR12B", "LTR12F", "LTR7Y", "HERVH-int", "LTR7", "LTR7B", "MER57-int", "MER61A")

candidate_results <- res_tbl %>%
  filter(feature_id %in% tcga_candidate_hits)

write_csv(candidate_results, file.path(results_dir, "MPNST_TCGA_candidate_ERV_hits_DESeq2_results.csv"))
candidate_results

In [ ]:
# Variance-stabilised expression for plots

vsd <- varianceStabilizingTransformation(dds, blind = FALSE)
vsd_mat <- assay(vsd)

write_csv(
  as_tibble(vsd_mat, rownames = "feature_id", .name_repair = "minimal"),
  file.path(results_dir, "MPNST_TE_vst_expression_matrix.csv")
)

# Save a small matrix for candidate ERV hits from the earlier TCGA analyses
candidate_vst <- vsd_mat[intersect(tcga_candidate_hits, rownames(vsd_mat)), , drop = FALSE]
write_csv(
  as_tibble(candidate_vst, rownames = "feature_id", .name_repair = "minimal"),
  file.path(results_dir, "MPNST_candidate_ERV_vst_expression_matrix.csv")
)

candidate_vst[, 1:min(5, ncol(candidate_vst)), drop = FALSE]

In [ ]:
# Annotate Features Using the Native SalmonTE Reference

clades_file <- file.path(
  mpnst_root,
  "software",
  "SalmonTE",
  "reference",
  "hs",
  "clades.csv"
)

if (!file.exists(clades_file)) {
  stop("Missing SalmonTE clade annotation: ", clades_file)
}

salmonte_annotation <- read_csv(
  clades_file,
  col_names = FALSE,
  show_col_types = FALSE
) %>%
  select(1:3) %>%
  setNames(c("feature_id", "SalmonTE_class", "SalmonTE_family")) %>%
  mutate(across(everything(), as.character)) %>%
  filter(
    !is.na(feature_id),
    feature_id != "",
    !tolower(feature_id) %in% c("name", "feature", "feature_id")
  ) %>%
  distinct(feature_id, .keep_all = TRUE)

cat("SalmonTE annotation rows:", nrow(salmonte_annotation), "\n")

# Reload the DESeq2 table -> cell works after a kernel restart
de_results_file <- file.path(
  results_dir,
  "MPNST_PRC2_loss_vs_intact_TE_DESeq2_results.csv"
)
if (!file.exists(de_results_file)) {
  stop("Missing DESeq2 results: ", de_results_file)
}

res_tbl_for_annotation <- read_csv(
  de_results_file,
  show_col_types = FALSE
)

# Join by the exact feature identifier generated by SalmonTE. 
annotated_results <- res_tbl_for_annotation %>%
  left_join(salmonte_annotation, by = "feature_id") %>%
  mutate(
    annotation_status = if_else(
      is.na(SalmonTE_family),
      "unmatched",
      "matched"
    ),
    is_ERV1 = if_else(
      is.na(SalmonTE_family),
      NA,
      SalmonTE_family == "ERV1"
    ),
    is_ERV2 = if_else(
      is.na(SalmonTE_family),
      NA,
      SalmonTE_family == "ERV2"
    ),
    significant_fdr_0_05 = !is.na(padj) & padj < 0.05,
    up_fdr_0_05 = significant_fdr_0_05 & log2FoldChange > 0,
    down_fdr_0_05 = significant_fdr_0_05 & log2FoldChange < 0
  )

annotated_results_file <- file.path(
  results_dir,
  "MPNST_PRC2_loss_vs_intact_TE_DESeq2_results_SalmonTE_annotated.csv"
)
write_csv(annotated_results, annotated_results_file)

write_csv(
  salmonte_annotation,
  file.path(results_dir, "SalmonTE_human_reference_clade_annotation.csv")
)

unmatched_annotation <- annotated_results %>%
  filter(annotation_status == "unmatched") %>%
  select(feature_id, baseMean, log2FoldChange, pvalue, padj)

write_csv(
  unmatched_annotation,
  file.path(results_dir, "MPNST_TE_SalmonTE_annotation_unmatched_features.csv")
)

annotation_coverage <- annotated_results %>%
  dplyr::count(annotation_status, name = "n_features") %>%
  mutate(percent = 100 * n_features / sum(n_features))

write_csv(
  annotation_coverage,
  file.path(results_dir, "MPNST_TE_SalmonTE_annotation_coverage_summary.csv")
)

cat("Annotated results written to:\n", annotated_results_file, "\n")
print(annotation_coverage)
cat("Unmatched tested features:", nrow(unmatched_annotation), "\n")

# Summarise FDR-significant features using SalmonTE's native class and family
significant_by_class <- annotated_results %>%
  filter(significant_fdr_0_05, !is.na(SalmonTE_class)) %>%
  dplyr::count(SalmonTE_class, sort = TRUE, name = "n_significant")

significant_by_family <- annotated_results %>%
  filter(significant_fdr_0_05, !is.na(SalmonTE_family)) %>%
  dplyr::count(SalmonTE_family, sort = TRUE, name = "n_significant")

write_csv(
  significant_by_class,
  file.path(results_dir, "MPNST_significant_TE_features_by_SalmonTE_class.csv")
)
write_csv(
  significant_by_family,
  file.path(results_dir, "MPNST_significant_TE_features_by_SalmonTE_family.csv")
)

significant_by_class
significant_by_family

In [ ]:
# ERV1 and ERV2 (ERVK) enrichment tests

# One-sided Fisher exact tests to determine whether a SalmonTE family is overrepresented among a specified set of differential-expression hits
run_family_enrichment <- function(
  data,
  target_column,
  hit_column,
  test_name,
  family_label,
  alternative = "greater"
) {
  target <- data[[target_column]]
  hit <- data[[hit_column]]

  # Unannotated rows are excluded rather than being counted as non-target TEs
  keep <- !is.na(target) & !is.na(hit)
  target <- target[keep]
  hit <- hit[keep]

  a <- sum(target & hit)
  b <- sum(target & !hit)
  c_count <- sum(!target & hit)
  d <- sum(!target & !hit)

  contingency_table <- matrix(
    c(a, b, c_count, d),
    nrow = 2,
    byrow = TRUE,
    dimnames = list(
      c(family_label, "Other_SalmonTE_families"),
      c("Hit", "Not_hit")
    )
  )

  fisher_result <- stats::fisher.test(
    contingency_table,
    alternative = alternative
  )

  tibble(
    test_name = test_name,
    family = family_label,
    alternative = alternative,
    n_annotated_features_tested = length(target),
    target_hits = a,
    target_not_hits = b,
    other_hits = c_count,
    other_not_hits = d,
    odds_ratio = unname(fisher_result$estimate),
    confidence_low = fisher_result$conf.int[1],
    confidence_high = fisher_result$conf.int[2],
    p_value = fisher_result$p.value
  )
}

enrichment_results <- bind_rows(
  run_family_enrichment(
    annotated_results, "is_ERV1", "significant_fdr_0_05",
    "ERV1_among_all_significant", "ERV1"
  ),
  run_family_enrichment(
    annotated_results, "is_ERV1", "up_fdr_0_05",
    "ERV1_among_upregulated", "ERV1"
  ),
  run_family_enrichment(
    annotated_results, "is_ERV1", "down_fdr_0_05",
    "ERV1_among_downregulated", "ERV1"
  ),
  run_family_enrichment(
    annotated_results, "is_ERV2", "down_fdr_0_05",
    "ERV2_HERVK_associated_among_downregulated", "ERV2"
  )
) %>%
  mutate(p_adjusted_BH = stats::p.adjust(p_value, method = "BH"))

write_csv(
  enrichment_results,
  file.path(results_dir, "MPNST_TE_SalmonTE_family_enrichment_Fisher_results.csv")
)

enrichment_results

In [ ]:
# Simple candidate violin plots

if (!requireNamespace("ggplot2", quietly = TRUE)) {
  stop("The ggplot2 package is required for this plotting cell.")
}
library(ggplot2)
library(tidyr)

candidate_features <- c(
  "MER9a3", "LTR54", "LTR10D", "LTR12",
  "LTR57", "LTR8", "MER57F", "HERVK"
)

vst_file <- file.path(results_dir, "MPNST_TE_vst_expression_matrix.csv")
if (!file.exists(vst_file)) stop("Missing VST expression matrix: ", vst_file)

vst_table <- read_csv(vst_file, show_col_types = FALSE)
if (!"feature_id" %in% colnames(vst_table)) {
  stop("The VST matrix does not contain a feature_id column.")
}

# Convert the wide expression matrix to one row per feature and sample
plot_metadata <- read_csv(sample_manifest_file, show_col_types = FALSE) %>%
  transmute(
    sample_id = as.character(sample_id),
    PRC2_status = factor(
      PRC2_status,
      levels = c("PRC2_intact", "PRC2_loss")
    )
  ) %>%
  distinct(sample_id, .keep_all = TRUE)

candidate_plot_data <- vst_table %>%
  filter(feature_id %in% candidate_features) %>%
  pivot_longer(
    cols = -feature_id,
    names_to = "sample_id",
    values_to = "vst_expression"
  ) %>%
  left_join(plot_metadata, by = "sample_id") %>%
  filter(!is.na(PRC2_status)) %>%
  mutate(feature_id = factor(feature_id, levels = candidate_features))

missing_candidates <- setdiff(
  candidate_features,
  unique(as.character(candidate_plot_data$feature_id))
)
if (length(missing_candidates) > 0) {
  warning(
    "Candidate features absent from VST matrix: ",
    paste(missing_candidates, collapse = ", ")
  )
}

# Load DESeq2 statistics annotated using SalmonTE's native reference 
annotated_results_for_plot_file <- file.path(
  results_dir,
  "MPNST_PRC2_loss_vs_intact_TE_DESeq2_results_SalmonTE_annotated.csv"
)
if (!file.exists(annotated_results_for_plot_file)) {
  stop("Missing SalmonTE-annotated DESeq2 results. Run Cell 8 first.")
}

annotated_results_for_plot <- read_csv(
  annotated_results_for_plot_file,
  show_col_types = FALSE
)

# Use exact scientific notation for small values instead of displaying values below 1e-4 using the same threshold label
format_plot_p <- function(x) {
  ifelse(
    is.na(x),
    "NA",
    ifelse(
      x < 0.001,
      formatC(x, format = "e", digits = 2),
      formatC(x, format = "f", digits = 3)
    )
  )
}

# Keep each panel annotation narrow enough to fit cleanly
candidate_plot_labels <- annotated_results_for_plot %>%
  filter(feature_id %in% candidate_features) %>%
  transmute(
    feature_id = factor(feature_id, levels = candidate_features),
    statistical_label = paste0(
      "log2FC = ", sprintf("%+.2f", log2FoldChange),
      "\nP = ", format_plot_p(pvalue),
      "\nFDR = ", format_plot_p(padj)
    )
  )

# Short group labels prevent overlap while retaining the sample numbers
status_counts <- candidate_plot_data %>%
  distinct(sample_id, PRC2_status) %>%
  dplyr::count(PRC2_status, name = "n")

status_axis_labels <- stats::setNames(
  paste0(
    ifelse(status_counts$PRC2_status == "PRC2_intact", "Intact", "Loss"),
    "\n(n = ", status_counts$n, ")"
  ),
  as.character(status_counts$PRC2_status)
)

write_csv(
  candidate_plot_data,
  file.path(results_dir, "MPNST_candidate_TE_violin_plot_data.csv")
)

candidate_violin_plot <- ggplot(
  candidate_plot_data,
  aes(x = PRC2_status, y = vst_expression, fill = PRC2_status)
) +
  geom_violin(
    trim = FALSE,
    width = 0.82,
    alpha = 0.32,
    colour = NA
  ) +
  geom_boxplot(
    width = 0.15,
    outlier.shape = NA,
    alpha = 0.60,
    linewidth = 0.35
  ) +
  geom_jitter(
    width = 0.07,
    height = 0,
    size = 1.4,
    alpha = 0.72
  ) +
  geom_text(
    data = candidate_plot_labels,
    aes(x = 1.5, y = Inf, label = statistical_label),
    inherit.aes = FALSE,
    vjust = 1.05,
    size = 2.65,
    lineheight = 0.92
  ) +
  facet_wrap(~ feature_id, scales = "free_y", ncol = 4) +
  scale_fill_manual(
    values = c("PRC2_intact" = "#8A8A8A", "PRC2_loss" = "#C44E52")
  ) +
  scale_x_discrete(labels = status_axis_labels) +
  scale_y_continuous(expand = expansion(mult = c(0.05, 0.34))) +
  labs(
    title = "Candidate TE expression in MPNST",
    subtitle = "PRC2-loss tumours compared with PRC2-intact tumours",
    x = NULL,
    y = "Variance-stabilised TE expression"
  ) +
  theme_classic(base_size = 11) +
  theme(
    legend.position = "none",
    strip.background = element_blank(),
    strip.text = element_text(face = "bold", size = 10),
    plot.title = element_text(face = "bold"),
    plot.subtitle = element_text(size = 10),
    axis.text.x = element_text(size = 9),
    panel.spacing.x = grid::unit(1.25, "lines"),
    panel.spacing.y = grid::unit(1.1, "lines"),
    plot.margin = margin(8, 12, 8, 8)
  )

candidate_violin_plot

ggsave(
  file.path(results_dir, "MPNST_candidate_TE_expression_violin_plots.png"),
  candidate_violin_plot,
  width = 12,
  height = 7.5,
  dpi = 300
)

ggsave(
  file.path(results_dir, "MPNST_candidate_TE_expression_violin_plots.pdf"),
  candidate_violin_plot,
  width = 12,
  height = 7.5
)

In [ ]:
# Volcano plot of all TE tested features

if (!requireNamespace("ggplot2", quietly = TRUE)) {
  stop("The ggplot2 package is required for this plotting cell.")
}
library(ggplot2)

annotated_results_file <- file.path(
  results_dir,
  "MPNST_PRC2_loss_vs_intact_TE_DESeq2_results_SalmonTE_annotated.csv"
)
if (!file.exists(annotated_results_file)) {
  stop("Missing SalmonTE-annotated results. Run Cell 8 first.")
}

volcano_data <- read_csv(
  annotated_results_file,
  show_col_types = FALSE
) %>%
  mutate(
    minus_log10_p = -log10(pmax(pvalue, .Machine$double.xmin)),
    result_group = case_when(
      !is.na(padj) & padj < 0.05 & log2FoldChange > 0 ~ "Higher in PRC2 loss",
      !is.na(padj) & padj < 0.05 & log2FoldChange < 0 ~ "Lower in PRC2 loss",
      TRUE ~ "Not FDR-significant"
    ),
    result_group = factor(
      result_group,
      levels = c(
        "Higher in PRC2 loss",
        "Lower in PRC2 loss",
        "Not FDR-significant"
      )
    )
  )

# Label the same representative features used in the violin-plot figure
volcano_label_features <- c(
  "MER9a3", "LTR54", "LTR10D", "LTR12",
  "LTR57", "LTR8", "MER57F", "HERVK"
)

volcano_labels <- volcano_data %>%
  filter(feature_id %in% volcano_label_features)

volcano_plot <- ggplot(
  volcano_data,
  aes(x = log2FoldChange, y = minus_log10_p, colour = result_group)
) +
  geom_hline(
    yintercept = -log10(0.05),
    linetype = "dashed",
    linewidth = 0.35,
    colour = "#777777"
  ) +
  geom_vline(
    xintercept = 0,
    linetype = "dotted",
    linewidth = 0.35,
    colour = "#999999"
  ) +
  geom_point(size = 1.7, alpha = 0.72) +
  scale_colour_manual(
    values = c(
      "Higher in PRC2 loss" = "#C44E52",
      "Lower in PRC2 loss" = "#4C78A8",
      "Not FDR-significant" = "#BDBDBD"
    ),
    drop = FALSE
  ) +
  labs(
    title = "TE differential expression in MPNST",
    subtitle = "PRC2-loss tumours compared with PRC2-intact tumours",
    x = "log2 fold-change",
    y = expression(-log[10](italic(P))),
    colour = NULL,
    caption = paste0(
      "Colours indicate Benjamini-Hochberg FDR < 0.05; ",
      "the dashed line marks nominal P = 0.05."
    )
  ) +
  theme_classic(base_size = 11) +
  theme(
    plot.title = element_text(face = "bold"),
    legend.position = "bottom",
    legend.box = "vertical",
    plot.caption = element_text(size = 8, hjust = 0)
  )

# ggrepel produces clearer non-overlapping labels when available
if (requireNamespace("ggrepel", quietly = TRUE)) {
  volcano_plot <- volcano_plot +
    ggrepel::geom_text_repel(
      data = volcano_labels,
      aes(label = feature_id),
      size = 3,
      colour = "#222222",
      box.padding = 0.35,
      point.padding = 0.25,
      min.segment.length = 0,
      max.overlaps = Inf,
      seed = 2026,
      show.legend = FALSE
    )
} else {
  volcano_plot <- volcano_plot +
    geom_text(
      data = volcano_labels,
      aes(label = feature_id),
      size = 2.8,
      colour = "#222222",
      vjust = -0.7,
      check_overlap = TRUE,
      show.legend = FALSE
    )
}

write_csv(
  volcano_data,
  file.path(results_dir, "MPNST_TE_volcano_plot_data.csv")
)

volcano_plot

ggsave(
  file.path(results_dir, "MPNST_TE_DESeq2_volcano_plot.png"),
  volcano_plot,
  width = 7.5,
  height = 6,
  dpi = 300
)

ggsave(
  file.path(results_dir, "MPNST_TE_DESeq2_volcano_plot.pdf"),
  volcano_plot,
  width = 7.5,
  height = 6
)

In [ ]:
# PCA quality control plot

if (!requireNamespace("ggplot2", quietly = TRUE)) {
  stop("The ggplot2 package is required for this plotting cell.")
}
library(ggplot2)

vst_file <- file.path(results_dir, "MPNST_TE_vst_expression_matrix.csv")
if (!file.exists(vst_file)) {
  stop("Missing VST expression matrix. Run Cell 7 first: ", vst_file)
}

vst_table_for_pca <- read_csv(vst_file, show_col_types = FALSE)
if (!"feature_id" %in% colnames(vst_table_for_pca)) {
  stop("The VST expression matrix does not contain feature_id.")
}

vst_matrix_for_pca <- vst_table_for_pca %>%
  column_to_rownames("feature_id") %>%
  as.matrix()

# PCA is calculated across samples using all 444 filtered TE features
pca_fit <- prcomp(
  t(vst_matrix_for_pca),
  center = TRUE,
  scale. = FALSE
)

variance_percent <- 100 * (pca_fit$sdev^2 / sum(pca_fit$sdev^2))

pca_data <- as_tibble(
  pca_fit$x[, 1:2, drop = FALSE],
  rownames = "sample_id"
) %>%
  left_join(
    read_csv(sample_manifest_file, show_col_types = FALSE) %>%
      transmute(
        sample_id = as.character(sample_id),
        PRC2_status = factor(
          PRC2_status,
          levels = c("PRC2_intact", "PRC2_loss")
        )
      ) %>%
      distinct(sample_id, .keep_all = TRUE),
    by = "sample_id"
  )

if (any(is.na(pca_data$PRC2_status))) {
  warning("Some PCA samples could not be assigned a PRC2 status.")
}

write_csv(
  pca_data,
  file.path(results_dir, "MPNST_TE_PCA_sample_coordinates.csv")
)

pca_plot <- ggplot(
  pca_data,
  aes(x = PC1, y = PC2, colour = PRC2_status, shape = PRC2_status)
) +
  geom_hline(yintercept = 0, colour = "#E5E5E5", linewidth = 0.3) +
  geom_vline(xintercept = 0, colour = "#E5E5E5", linewidth = 0.3) +
  geom_point(size = 3, alpha = 0.82) +
  scale_colour_manual(
    values = c("PRC2_intact" = "#777777", "PRC2_loss" = "#C44E52"),
    labels = c("PRC2 intact", "PRC2 loss")
  ) +
  scale_shape_manual(
    values = c("PRC2_intact" = 16, "PRC2_loss" = 17),
    labels = c("PRC2 intact", "PRC2 loss")
  ) +
  labs(
    title = "Principal-component analysis of MPNST TE expression",
    subtitle = "Variance-stabilised expression of 444 SalmonTE features",
    x = paste0("PC1 (", sprintf("%.1f", variance_percent[1]), "% variance)"),
    y = paste0("PC2 (", sprintf("%.1f", variance_percent[2]), "% variance)"),
    colour = NULL,
    shape = NULL
  ) +
  coord_equal() +
  theme_classic(base_size = 11) +
  theme(
    plot.title = element_text(face = "bold"),
    legend.position = "bottom"
  )

pca_plot

ggsave(
  file.path(results_dir, "MPNST_TE_PCA_PRC2_status.png"),
  pca_plot,
  width = 7,
  height = 5.8,
  dpi = 300
)

ggsave(
  file.path(results_dir, "MPNST_TE_PCA_PRC2_status.pdf"),
  pca_plot,
  width = 7,
  height = 5.8
)

In [ ]:
# Supplementary table for appendix

significant_mpnst <- annotated_results %>%
  filter(padj < 0.05) %>%
  select(
    feature_id,
    SalmonTE_class,
    SalmonTE_family,
    baseMean,
    log2FoldChange,
    lfcSE,
    pvalue,
    padj
  ) %>%
  arrange(padj)

write_csv(
  significant_mpnst,
  file.path(
    results_dir,
    "MPNST_FDR_significant_TE_features.csv"
  )
)